In [ ]:
# %% [0] Geometry, Mesh & Material Configuration  --  RF cross-section, Cordoba + SiO2 buffer + ALD Al2O3
#
#   Full two-arm GSG (push-pull) Cordoba cross-section at 60 GHz: centred signal, a rib with its
#   SiO2 cap in each gap, both grounds, unetched LN slab across the domain.
#   Optional layers, drawn exactly as in the optical (Vpi*L / IL / group-index) models:
#     BUFFER_H : SiO2 buffer on the whole slab, under the gold and across the gaps (merges with the
#                caps); the gold sits on it, GAP is measured there, CAP_H counts from its top.
#     T_ALD    : conformal ALD Al2O3 on every surface exposed to air after the electrodes
#                (electrode tops / walls, slab or buffer in the gaps and outside, cap sides / top).
import time
import warnings
from collections import OrderedDict

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from shapely.affinity import scale as _scale
from shapely.geometry import MultiPolygon, Polygon, box
from shapely.ops import unary_union
from skfem import MeshTri
from femwell.mesh import mesh_from_OrderedDict

warnings.filterwarnings("ignore")

# femwell bug fix (as in the optical notebooks): mesh_from_OrderedDict() never really switches
# off gmsh's "extend mesh size from boundary", so every fine surface size (e.g. the 20 nm ALD)
# propagates deep into the neighbouring regions -- here, into the PEC electrodes.
import gmsh
if not getattr(gmsh.model.mesh.field, "_size_fix", False):
    _orig_set_bg = gmsh.model.mesh.field.setAsBackgroundMesh

    def _set_bg_fixed(tag):
        _orig_set_bg(tag)
        gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
        gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
        gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)

    gmsh.model.mesh.field.setAsBackgroundMesh = _set_bg_fixed
    gmsh.model.mesh.field._size_fix = True

# ============================================================
# 1. PARAMETER BLOCK  (Cordoba)
# ============================================================
f0        = 60.0e9      # Hz    mode-analysis frequency
device_l  = 16.5e-3     # m     line length used for the S21 report

auc_w     = 35.0        # um    W_S  central signal electrode width
gap       = 4.15        # um    GAP  signal-to-ground gap (on the buffer when BUFFER_H > 0)
au_h      = 2.0         # um    MTX  electrode thickness
aul_w     = 49.85       # um    lateral ground width
cap_w     = 2.0         # um    SiO2 cap width
cap_h     = 1.40        # um    SiO2 cap height (from the buffer top)

tfln      = 0.540       # um    total thin-film LN thickness
wg_h      = 0.270       # um    rib height (etch depth)
slab_h    = tfln - wg_h # um    un-etched slab
wg_top    = 1.0         # um    rib top width
theta     = 60.0        # deg   rib sidewall angle

box_h     = 4.7         # um    buried SiO2
si_h      = 550.0       # um    silicon substrate (real thickness)
air_h     = 550.0       # um    air above the slab
lat_pad   = 200.0       # um    lateral padding beyond each outer ground edge

BUFFER_H  = 0.0         # um    SiO2 buffer (0 = none; 0.200 = buffered design)
T_ALD     = 0.0         # um    ALD Al2O3 thickness (0 = none; 0.040 = nominal)
EPS_ALD   = 9.8         #       ALD Al2O3 relative permittivity at RF (same value as the Vpi*L studies)
TAN_ALD   = 0.0         #       ALD Al2O3 loss tangent at f0 (0 = lossless; see the loss assessment)

basetta   = wg_h / np.tan(np.deg2rad(theta))
wg_bottom = wg_top + 2.0 * basetta

r9     = lambda v: round(float(v), 9)
y_b    = r9(slab_h + BUFFER_H)  # bottom of the gold (= buffer top)
x_sig  = auc_w / 2.0            # signal spans [-x_sig, +x_sig]
x_gi   = x_sig + gap            # ground inner edge
x_go   = x_gi + aul_w           # ground outer edge
x_dom  = x_go + lat_pad         # domain half width
x_wg   = x_sig + gap / 2.0      # rib centre (mid-gap)
y_el   = r9(y_b + au_h)         # electrode top
y_rib  = slab_h + wg_h          # rib top
y_cap  = r9(y_b + cap_h)        # cap top

# ============================================================
# 2. MATERIALS  (60 GHz)
# ============================================================
C0   = 299792458.0
EPS0 = 8.8541878128e-12
MU0  = 4.0e-7 * np.pi
ETA0 = np.sqrt(MU0 / EPS0)
DB_PER_NEPER = 20.0 * np.log10(np.e)

SIGMA_AU  = 4.56e7
skin_au   = np.sqrt(2.0 / (2 * np.pi * f0 * MU0 * SIGMA_AU))
Rs_au     = 1.0 / (SIGMA_AU * skin_au)
SIGMA_ALD = 2 * np.pi * f0 * EPS0 * EPS_ALD * TAN_ALD      # S/m equivalent of the loss tangent

MATERIALS = {
    "LN":    {"color": "#2ecc71", "eps_rf": (28.0, 44.0, 44.0), "sigma": 1e-3,  "desc": "LiNbO3 slab + ribs"},
    "SiO2":  {"color": "#00ebfc", "eps_rf": (3.9, 3.9, 3.9),    "sigma": 1e-13, "desc": "SiO2 box + caps + buffer"},
    "Al2O3": {"color": "#f472d0", "eps_rf": (EPS_ALD,) * 3,     "sigma": SIGMA_ALD, "desc": "ALD Al2O3"},
    "Si":    {"color": "#7f8c8d", "eps_rf": (11.7, 11.7, 11.7), "sigma": 1e-12, "desc": "Silicon substrate"},
    "air":   {"color": "#ffffff", "eps_rf": (1.0, 1.0, 1.0),    "sigma": 0.0,   "desc": "Air / vacuum"},
    "Au":    {"color": "#f39c12", "eps_rf": (1.0, 1.0, 1.0),    "sigma": 0.0,   "desc": "Gold electrodes (PEC)"},
}
PREFIX_TO_MAT = [("el_", "Au"), ("rib", "LN"), ("slab", "LN"), ("cap", "SiO2"), ("buf", "SiO2"),
                 ("box", "SiO2"), ("alu", "Al2O3"), ("si", "Si"), ("air", "air")]
METAL_REGIONS = ("el_sig", "el_gr", "el_gl")
SIGNAL_REGION = "el_sig"


def mat_of_region(name):
    base = name.split("__")[-1] if "__" in name else name
    for p, m in PREFIX_TO_MAT:
        if base.startswith(p):
            return m
    raise KeyError(name)


def mirror(p):
    return _scale(p, xfact=-1.0, yfact=1.0, origin=(0, 0))


def _areal(g, tol=1e-10):
    if g.is_empty:
        return Polygon()
    parts = [g] if isinstance(g, Polygon) else [q for q in getattr(g, "geoms", []) if isinstance(q, (Polygon, MultiPolygon))]
    parts = [q for p in parts for q in (p.geoms if isinstance(p, MultiPolygon) else [p]) if q.area > tol]
    return Polygon() if not parts else (parts[0] if len(parts) == 1 else MultiPolygon(parts))


# ============================================================
# 3. MESH CONTROL
# ============================================================
MESH_FACTOR = 1.0
COLLAR_W    = 4.0      # um, lateral reach of the fine air collar past the gap
COLLAR_H    = 4.0      # um, height of the fine air collar above the electrodes
CORNER_BOX  = 0.25     # um, half-size of the refinement box on each electrode corner
CORNER_RES  = 0.050    # um, mesh size inside those boxes
SKIN_T      = 0.10     # um, thickness of the conductor-surface collar
SKIN_REACH  = 5.0      # um, how far that collar runs along the top / bottom faces
SKIN_RES    = 0.100    # um, mesh size inside the collar
NEAR        = 15.0     # um, reach of the "near" zones past the outer ground edge


def build_polygons():
    def rib(xc):
        return Polygon([(xc - wg_bottom / 2, slab_h), (xc - wg_top / 2, y_rib),
                        (xc + wg_top / 2, y_rib), (xc + wg_bottom / 2, slab_h)])

    rib_r = rib(x_wg); rib_l = mirror(rib_r)
    cap_r = box(x_wg - cap_w / 2, slab_h, x_wg + cap_w / 2, y_cap).difference(rib_r); cap_l = mirror(cap_r)
    el_sig = box(-x_sig, y_b, x_sig, y_el)
    el_gr = box(x_gi, y_b, x_go, y_el); el_gl = mirror(el_gr)
    metals = unary_union([el_sig, el_gr, el_gl])
    buf = Polygon()
    if BUFFER_H > 0:
        buf = _areal(box(-x_dom, slab_h, x_dom, y_b).difference(unary_union([rib_r, rib_l, cap_r, cap_l])))
    solids = unary_union([metals, cap_r, cap_l, rib_r, rib_l, buf])
    slab = box(-x_dom, 0.0, x_dom, slab_h)
    alu = Polygon()
    if T_ALD > 0:
        shell = unary_union([solids, slab]).buffer(T_ALD, join_style=2, mitre_limit=10.0)
        alu = _areal(shell.intersection(box(-x_dom, slab_h, x_dom, slab_h + air_h)).difference(solids))
    air = _areal(box(-x_dom, slab_h, x_dom, slab_h + air_h).difference(unary_union([solids, alu])))

    # material zones (near / far)
    near = box(-x_go - NEAR, -box_h, x_go + NEAR, y_el + NEAR)
    gapz = box(-x_gi - COLLAR_W, slab_h, x_gi + COLLAR_W, y_el + COLLAR_H)
    zones = OrderedDict()
    zones["cap_r"], zones["cap_l"], zones["rib_r"], zones["rib_l"] = cap_r, cap_l, rib_r, rib_l
    zones["el_sig"], zones["el_gr"], zones["el_gl"] = el_sig, el_gr, el_gl
    zones["alu_near"] = _areal(alu.intersection(near)); zones["alu_far"] = _areal(alu.difference(near))
    zones["buf_near"] = _areal(buf.intersection(near)); zones["buf_far"] = _areal(buf.difference(near))
    zones["air_gap"] = _areal(air.intersection(gapz))
    zones["air_near"] = _areal(air.intersection(near).difference(gapz))
    zones["air_far"] = _areal(air.difference(near))
    zones["slab_near"] = _areal(slab.intersection(near)); zones["slab_far"] = _areal(slab.difference(near))
    zones["box_near"] = box(-x_go - NEAR, -box_h, x_go + NEAR, 0.0)
    zones["box_far"] = _areal(box(-x_dom, -box_h, x_dom, 0.0).difference(zones["box_near"]))
    zones["si"] = box(-x_dom, -box_h - si_h, x_dom, -box_h)

    # conductor-surface refinement collars (a meshing device): cut into every
    # non-metal zone they cross, so each piece keeps the material of its zone
    corners = [box(xc - CORNER_BOX, yc - CORNER_BOX, xc + CORNER_BOX, yc + CORNER_BOX)
               for xc in (-x_sig, x_sig, -x_gi, x_gi, -x_go, x_go) for yc in (y_b, y_el)]
    corner_ring = unary_union(corners).difference(metals)
    walls = []
    for xa, xb in ((x_sig, x_sig + SKIN_T), (-x_sig - SKIN_T, -x_sig), (x_gi - SKIN_T, x_gi), (-x_gi, -x_gi + SKIN_T)):
        walls.append(box(xa, y_b, xb, y_el))
    for xc in (-x_sig, x_sig, -x_gi, x_gi):
        lo, hi = (xc, xc + SKIN_REACH) if xc > 0 else (xc - SKIN_REACH, xc)
        walls.append(box(lo, y_el, hi, y_el + SKIN_T))
        walls.append(box(lo, y_b - SKIN_T, hi, y_b))
    skin_ring = unary_union(walls).difference(unary_union([metals, corner_ring]))

    polys = OrderedDict()
    for ring_name, ring in (("corner_ring", corner_ring), ("skin_ring", skin_ring)):
        for zn, z in zones.items():
            if zn.startswith("el_") or z.is_empty:
                continue
            piece = _areal(ring.intersection(z))
            if not piece.is_empty:
                polys[f"{ring_name}__{zn}"] = piece
    rings = unary_union([corner_ring, skin_ring])
    for zn, z in zones.items():
        z = z if zn.startswith("el_") else _areal(z.difference(rings))
        if not z.is_empty:
            polys[zn] = z
    import shapely
    polys = OrderedDict((k, v) for k, v in polys.items())
    return polys


BASE_RES = {   # um
    "cap_r": 0.20, "cap_l": 0.20, "rib_r": 0.12, "rib_l": 0.12,
    "air_gap": 0.30, "slab_near": 0.25, "box_near": 2.0,
    "el_sig": 0.80, "el_gr": 0.80, "el_gl": 0.80,
    "air_near": 2.0, "slab_far": 6.0, "box_far": 6.0, "si": 30.0, "air_far": 30.0,
    "buf_near": min(BUFFER_H / 3, 0.10) if BUFFER_H > 0 else 0.1, "buf_far": 1.0,
    "alu_near": min(T_ALD / 2, 0.05) if T_ALD > 0 else 0.05, "alu_far": 1.0,
}
BASE_DIST = {
    "cap_r": 1.0, "cap_l": 1.0, "rib_r": 1.0, "rib_l": 1.0, "air_gap": 3.0, "slab_near": 3.0,
    "box_near": 5.0, "el_sig": 3.0, "el_gr": 3.0, "el_gl": 3.0, "air_near": 10.0, "slab_far": 10.0,
    "box_far": 10.0, "si": 30.0, "air_far": 30.0, "buf_near": 0.5, "buf_far": 2.0, "alu_near": 0.3, "alu_far": 2.0,
}
FIXED = ("corner_ring", "skin_ring", "alu_near", "buf_near")   # not scaled by MESH_FACTOR (geometry-set)


def _res(k):
    if k.startswith("corner_ring"):
        zn = k.split("__")[1]; return min(CORNER_RES, BASE_RES[zn]), 0.6
    if k.startswith("skin_ring"):
        zn = k.split("__")[1]; return min(SKIN_RES, BASE_RES[zn]), 1.0
    f = 1.0 if k.startswith(FIXED) else MESH_FACTOR
    return BASE_RES[k] * f, BASE_DIST[k]


polygons = build_polygons()
resolutions = {k: {"resolution": _res(k)[0], "distance": _res(k)[1]} for k in polygons}

# validity: the regions must tile the domain exactly, without overlaps
_tot = sum(p.area for p in polygons.values())
assert abs(_tot - 2 * x_dom * (box_h + si_h + slab_h + air_h)) < 1e-4 * _tot, "tiling gap"

t_mesh = time.time()
raw_mesh = mesh_from_OrderedDict(polygons, resolutions=resolutions,
                                 default_resolution_min=0.4 * min(CORNER_RES, SKIN_RES, BASE_RES["alu_near"]),
                                 default_resolution_max=35.0)
mesh = MeshTri(raw_mesh.points[:, :2].T, raw_mesh.cells_dict["triangle"].T)
t_mesh = time.time() - t_mesh

tri_subdomains = raw_mesh.cell_data_dict["gmsh:physical"]["triangle"]
name_to_id = {n: (d[0] if hasattr(d, "__getitem__") else d) for n, d in raw_mesh.field_data.items()}
region_of = np.empty(mesh.nelements, dtype=object)
for raw_name, s_id in name_to_id.items():
    base_name = raw_name.split("___")[0]
    if base_name in polygons:
        region_of[tri_subdomains == s_id] = base_name
assert not any(r is None for r in region_of), "untagged elements in the mesh"
mat_of_el = np.array([mat_of_region(r) for r in region_of], dtype=object)

is_metal  = np.isin(region_of.astype(str), METAL_REGIONS)
is_signal = region_of.astype(str) == SIGNAL_REGION

f2t          = mesh.f2t
has_two      = f2t[1] >= 0
on_contour   = has_two & (is_metal[f2t[0]] ^ is_metal[f2t[1]])
metal_side   = np.where(is_metal[f2t[0]], f2t[0], f2t[1])
contour_sig  = on_contour & is_signal[metal_side]
contour_gnd  = on_contour & ~is_signal[metal_side]
contour_all  = on_contour

print("=" * 88)
print(f"{'MATERIAL':<34} | {'eps_rf (xx, yy, zz)':<24} | {'sigma [S/m]':<12}")
print("-" * 88)
for key, d in MATERIALS.items():
    e = d["eps_rf"]
    print(f"{d['desc']:<34} | {f'({e[0]:.1f}, {e[1]:.1f}, {e[2]:.1f})':<24} | {d['sigma']:<12.3g}")
print("=" * 88)
print(f"f0 = {f0/1e9:.1f} GHz | buffer {BUFFER_H*1e3:.0f} nm | ALD {T_ALD*1e3:.0f} nm (eps {EPS_ALD}, tan d {TAN_ALD})")
print(f"domain  {2*x_dom:.1f} x {si_h+box_h+slab_h+air_h:.1f} um   (lateral padding {lat_pad:.0f} um, air {air_h:.0f} um, Si {si_h:.0f} um)")
print(f"mesh    {mesh.nelements} triangles, {mesh.nvertices} nodes, built in {t_mesh:.1f} s")


def contour_groups(mask):
    out = []
    for side in (0, 1):
        other = 1 - side
        fac = np.where(mask & is_metal[f2t[other]] & ~is_metal[f2t[side]])[0]
        if fac.size == 0:
            continue
        p = mesh.p[:, mesh.facets[:, fac]]
        tan = p[:, 1, :] - p[:, 0, :]
        nrm = np.vstack([tan[1], -tan[0]])
        nrm /= np.linalg.norm(nrm, axis=0)
        midpoint = p.mean(axis=1)
        centroid = mesh.p[:, mesh.t[:, f2t[other][fac]]].mean(axis=1)
        nrm *= np.sign(((midpoint - centroid) * nrm).sum(axis=0))
        out.append((fac, side, nrm))
    return out


CONTOUR = {"sig": contour_groups(contour_sig), "gnd": contour_groups(contour_gnd), "all": contour_groups(contour_all)}

# ---- cross-section (materials, from the mesh) ------------------------------
SHOW_MESH = True
from matplotlib.colors import ListedColormap
_mk = list(MATERIALS)
_mi = np.array([_mk.index(m) for m in mat_of_el], dtype=float)
_cm = ListedColormap([MATERIALS[m]["color"] for m in _mk])
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
for ax, (xl, yl, ttl) in zip(axes, [((x_sig - 1.5, x_gi + 1.5), (-0.6, y_el + 1.5), "Right gap"),
                                    ((x_gi - 0.5, x_gi + 0.5), (y_b - 0.35, y_b + 0.45), "Ground inner corner (zoom)"),
                                    ((-x_dom, x_dom), (-box_h - 40, 60), "Full domain")]):
    ax.tripcolor(mesh.p[0], mesh.p[1], mesh.t.T, facecolors=_mi, cmap=_cm, vmin=-0.5, vmax=len(_mk) - 0.5)
    if SHOW_MESH and not ttl.startswith("Full"):
        mesh.draw(ax=ax, color="black", lw=0.1)
    ax.set_xlim(xl); ax.set_ylim(yl); ax.set_aspect("equal" if not ttl.startswith("Full") else "auto")
    ax.set_title(ttl); ax.set_xlabel(r"$x$ ($\mu$m)")
axes[0].legend(handles=[mpatches.Patch(color=MATERIALS[m]["color"], label=m) for m in _mk if m in set(mat_of_el)],
               fontsize=7, loc="upper right")
fig.suptitle(f"Cordoba RF cross-section: signal {auc_w}, gap {gap}, grounds {aul_w}, Au {au_h}, LN {tfln} (rib {wg_h}), "
             f"cap {cap_w} x {cap_h}, buffer {BUFFER_H*1e3:.0f} nm, ALD {T_ALD*1e3:.0f} nm", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# %% [1] Anisotropic Quasi-Static Extraction (independent n_m, Z0 and the eigensolver shift)
from skfem import (Basis, BilinearForm, ElementTriP0, ElementTriP1, Functional,
                   InteriorFacetBasis, asm, condense, solve)
from skfem.helpers import grad

basis_qs = Basis(mesh, ElementTriP1())
basis_p0 = Basis(mesh, ElementTriP0())

eps_x = np.array([MATERIALS[m]["eps_rf"][0] for m in mat_of_el], dtype=float)
eps_y = np.array([MATERIALS[m]["eps_rf"][1] for m in mat_of_el], dtype=float)
eps_z = np.array([MATERIALS[m]["eps_rf"][2] for m in mat_of_el], dtype=float)
sigma = np.array([MATERIALS[m]["sigma"] for m in mat_of_el], dtype=float)
ones  = np.ones(mesh.nelements)


@BilinearForm
def laplace_aniso(u, v, w):
    return w.ex * grad(u)[0] * grad(v)[0] + w.ey * grad(u)[1] * grad(v)[1]


# every DOF inside the gold is pinned: the electrodes are equipotential bodies
dofs_sig = np.unique(basis_qs.element_dofs[:, np.where(is_signal)[0]])
dofs_gnd = np.unique(basis_qs.element_dofs[:, np.where(is_metal & ~is_signal)[0]])
dofs_D   = np.unique(np.concatenate([dofs_sig, dofs_gnd]))


def solve_static(ex, ey):
    """Laplace with V = 1 V on the signal, 0 V on both grounds."""
    K = asm(laplace_aniso, basis_qs,
            ex=basis_p0.interpolate(ex), ey=basis_p0.interpolate(ey))
    u_D = np.zeros(basis_qs.N)
    u_D[dofs_sig] = 1.0
    Kc, fc, uc, I = condense(K, np.zeros(basis_qs.N), x=u_D, D=dofs_D)
    u = uc.copy()
    u[I] = solve(Kc, fc)
    # variational charge recovery: Q/eps0 = sum of the residual over the pinned DOFs.
    # This is exact for the discrete solution, unlike integrating dV/dn over a
    # contour that runs through the re-entrant corners of the electrodes.
    return u, EPS0 * float((K @ u)[dofs_sig].sum())


t_qs = time.time()
V_eps, C_eps = solve_static(eps_x, eps_y)      # F/m, the um -> m factors cancel
V_air, C_air = solve_static(ones, ones)

n_m_qs = np.sqrt(C_eps / C_air)
Z0_qs  = 1.0 / (C0 * np.sqrt(C_eps * C_air))
L_qs   = 1.0 / (C0 ** 2 * C_air)


# ---- Wheeler perturbation loss from the air-problem surface charge ----------
# For a quasi-TEM line  Jsz(l)/I = rho_air(l)/Q_air, so  R' = Rs * contour(|Jsz/I|^2).
@Functional
def dVdn_squared(w):
    return (w.gu.grad[0] * w.nx + w.gu.grad[1] * w.ny) ** 2


def contour_static(key, field):
    total = 0.0
    for fac, side, nrm in CONTOUR[key]:
        fb = InteriorFacetBasis(mesh, ElementTriP1(), facets=fac, side=side)
        total += dVdn_squared.assemble(fb, gu=fb.interpolate(field),
                                       nx=nrm[0][:, None], ny=nrm[1][:, None])
    return total


S_charge   = C_air / EPS0                          # = contour integral of dV/dn on the signal
R_prime_qs = Rs_au * 1e6 * contour_static("all", V_air) / S_charge ** 2
alpha_qs   = R_prime_qs / (2.0 * Z0_qs)            # Np/m
alpha_qs_db = DB_PER_NEPER * alpha_qs / 100.0
t_qs = time.time() - t_qs

print("=" * 68)
print("  QUASI-STATIC EXTRACTION  (Laplace with the anisotropic eps_RF tensor)")
print("=" * 68)
print(f"  C                 {C_eps*1e12:12.4f} pF/m")
print(f"  C_air             {C_air*1e12:12.4f} pF/m")
print(f"  L                 {L_qs*1e9:12.4f} nH/m")
print(f"  n_m               {n_m_qs:12.5f}")
print(f"  Z0                {Z0_qs:12.4f} ohm")
print(f"  R' (perturbation) {R_prime_qs:12.2f} ohm/m")
print(f"  alpha_c (perturb) {alpha_qs_db:12.4f} dB/cm")
print(f"  solved in         {t_qs:12.2f} s")
print("=" * 68)

n_guess = n_m_qs      # eigensolver shift; COMSOL used a flat 1.95 for every geometry

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
mesh.plot(V_eps, ax=axes[0], shading="gouraud", cmap="coolwarm", colorbar=True)
axes[0].set_title(r"$V(x,y)$ with $\epsilon_{RF}$ = diag(28, 44)")
mesh.plot(V_air, ax=axes[1], shading="gouraud", cmap="coolwarm", colorbar=True)
axes[1].set_title(r"$V_{air}(x,y)$  ($\epsilon$ = 1, sets $L$ and $J_{sz}$)")
for ax in axes:
    ax.set_xlim([-x_gi - 6, x_gi + 6]); ax.set_ylim([-2.0, au_h + 3.0])
    ax.set_aspect("equal"); ax.set_xlabel(r"$x$ ($\mu$m)"); ax.set_ylabel(r"$y$ ($\mu$m)")
plt.tight_layout(); plt.show()

In [ ]:
# %% [2] RF Vector Eigensolver (anisotropic, PEC electrodes = COMSOL IBC edges)
import scipy.constants
from skfem import BilinearForm, ElementTriN1, ElementTriP0, ElementTriP1, condense, solve
from skfem.helpers import curl, dot, grad
from skfem.utils import solver_eigen_scipy

NUM_MODES = 20           # COMSOL asked ARPACK for 40; the filters below need far fewer
SHIFT     = n_guess      # "search for modes around shift"; COMSOL used a flat 1.95

wl_rf = C0 / f0 * 1e6                 # um
k0    = 2.0 * np.pi / wl_rf           # 1/um
omega = 2.0 * np.pi * f0
UM    = 1e-6

element_rf = ElementTriN1() * ElementTriP1()
basis_rf   = basis_p0.with_element(element_rf)
basis_eps  = basis_rf.with_element(ElementTriP0())


@BilinearForm(dtype=complex)
def aform(e_t, e_z, v_t, v_z, w):
    return (curl(e_t) * curl(v_t) / k0 ** 2
            - (w.exx * e_t[0] * v_t[0] + w.eyy * e_t[1] * v_t[1])
            + dot(grad(e_z), v_t)
            + (w.exx * e_t[0] * grad(v_z)[0] + w.eyy * e_t[1] * grad(v_z)[1])
            - w.ezz * e_z * v_z * k0 ** 2)


@BilinearForm(dtype=complex)
def bform(e_t, e_z, v_t, v_z, w):
    return -dot(e_t, v_t) / k0 ** 2


t_asm = time.time()
A = aform.assemble(basis_rf, exx=basis_eps.interpolate(eps_x),
                   eyy=basis_eps.interpolate(eps_y), ezz=basis_eps.interpolate(eps_z))
B = bform.assemble(basis_rf)
t_asm = time.time() - t_asm

# PEC everywhere the field must vanish: inside + on the gold, and on the outer window
dofs_metal = np.unique(basis_rf.element_dofs[:, np.where(is_metal)[0]])
dofs_outer = basis_rf.get_dofs(facets=mesh.boundary_facets()).flatten()
dofs_pec   = np.unique(np.concatenate([dofs_metal, dofs_outer]))

t_eig = time.time()
lams, xs = solve(*condense(-A, -B, D=dofs_pec, x=basis_rf.zeros(dtype=complex)),
                 solver=solver_eigen_scipy(k=NUM_MODES, sigma=k0 ** 2 * SHIFT ** 2))
t_eig = time.time() - t_eig

# femwell scaling: the solved z-unknown is (j*beta/k0^2) * E_z
idx_t, idx_z = basis_rf.split_indices()
xs[idx_z, :] /= 1j * np.sqrt(lams[np.newaxis, :] / k0 ** 4)

betas = np.sqrt(lams.astype(complex))          # 1/um
neffs = betas / k0

print(f"assembled {A.shape[0]} DOF in {t_asm:.1f} s   |   "
      f"{NUM_MODES} modes around n = {SHIFT:.4f} in {t_eig:.1f} s")

In [ ]:
# %% [3] Mode Diagnostics and the COMSOL Three-Filter Pipeline
from skfem import Functional, InteriorFacetBasis

# the residual of the Gauss row over the pinned gold DOFs is the conductor charge;
# charge conservation on the contour then gives the longitudinal current exactly:
#   j*beta*Isig + d(Jst)/dl + j*omega*Qsig = 0   ->   Isig = (omega/beta) * Qsig
xs_raw = xs.copy()
xs_raw[idx_z, :] *= 1j * np.sqrt(lams[np.newaxis, :] / k0 ** 4)

_is_z = np.zeros(basis_rf.N, dtype=bool); _is_z[idx_z] = True
dofs_sig_z = np.unique(basis_rf.element_dofs[:, np.where(is_signal)[0]])
dofs_sig_z = dofs_sig_z[_is_z[dofs_sig_z]]
dofs_gnd_z = np.unique(basis_rf.element_dofs[:, np.where(is_metal & ~is_signal)[0]])
dofs_gnd_z = dofs_gnd_z[_is_z[dofs_gnd_z]]


@Functional(dtype=complex)
def f_poynting(w):                 # COMSOL emw.Poavz over every non-metal domain
    Hx = (w.ez.grad[1] / UM - 1j * w.beta * w.et[1]) / (-1j * omega * MU0)
    Hy = (1j * w.beta * w.et[0] - w.ez.grad[0] / UM) / (-1j * omega * MU0)
    return 0.5 * np.real(w.et[0] * np.conj(Hy) - w.et[1] * np.conj(Hx)) * UM ** 2


@Functional(dtype=complex)
def f_dielectric(w):
    return 0.5 * w.sig * (np.abs(w.et[0]) ** 2 + np.abs(w.et[1]) ** 2
                          + np.abs(w.ez) ** 2) * UM ** 2


@Functional(dtype=complex)
def f_Js2(w):                      # |Js|^2 = |Jsz|^2 + |Hz|^2 , the IBC loss integrand
    dEz_dn = (w.ez.grad[0] * w.nx + w.ez.grad[1] * w.ny) / UM
    E_n = w.et[0] * w.nx + w.et[1] * w.ny
    Jz = (-dEz_dn + 1j * w.beta * E_n) / (-1j * omega * MU0)
    Hz = (w.et.curl / UM) / (-1j * omega * MU0)
    return (np.abs(Jz) ** 2 + np.abs(Hz) ** 2) * UM


@Functional(dtype=complex)
def f_Jsz(w):
    dEz_dn = (w.ez.grad[0] * w.nx + w.ez.grad[1] * w.ny) / UM
    E_n = w.et[0] * w.nx + w.et[1] * w.ny
    return (-dEz_dn + 1j * w.beta * E_n) / (-1j * omega * MU0) * UM


def contour_mode(form, key, x, beta_m):
    total = 0.0 + 0j
    for fac, side, nrm in CONTOUR[key]:
        fb = InteriorFacetBasis(mesh, element_rf, facets=fac, side=side)
        fields = fb.interpolate(x)
        total += form.assemble(fb, et=fields[0], ez=fields[1],
                               nx=nrm[0][:, None], ny=nrm[1][:, None], beta=beta_m)
    return total


# COMSOL lineop1: integral of Ex across the gap at mid-electrode height
_xpath = np.linspace(-x_gi, -x_sig, 801)
_probe = basis_rf.split(np.zeros(basis_rf.N))[0][1].probes(
    np.vstack([_xpath, np.full_like(_xpath, y_b + au_h / 2.0)]))


def gap_voltage(x):
    Ex = np.asarray(_probe @ x[idx_t]).reshape(2, -1)[0]
    return np.trapezoid(Ex, _xpath) * UM


t_diag = time.time()
rows = []
for i in range(NUM_MODES):
    beta_m = betas[i] * 1e6                                  # rad/m
    if abs(beta_m) < 1e-9:
        continue
    x, xr = xs[:, i], xs_raw[:, i]
    fields = basis_rf.interpolate(x)
    P = abs(float(np.real(f_poynting.assemble(basis_rf, et=fields[0], ez=fields[1],
                                              beta=beta_m))))
    if P < 1e-30:
        continue
    res = A @ xr
    Q_sig = EPS0 * abs(res[dofs_sig_z].sum()) * 1e-6
    Q_gnd = EPS0 * abs(res[dofs_gnd_z].sum()) * 1e-6
    I_sig = omega / abs(beta_m) * Q_sig
    I_gnd = omega / abs(beta_m) * Q_gnd
    V_gap = abs(gap_voltage(x))
    Z_PI = 2.0 * P / I_sig ** 2 if I_sig > 0 else np.inf
    Z_VP = V_gap ** 2 / (2.0 * P)
    score = I_sig / (I_sig + I_gnd) if (I_sig + I_gnd) > 0 else np.nan
    div = abs(Z_PI - Z_VP) / max(0.5 * (Z_PI + Z_VP), 1e-30)
    rows.append(dict(index=i, neff=complex(neffs[i]), P=P, I_sig=I_sig, I_gnd=I_gnd,
                     V_gap=V_gap, Z_PI=Z_PI, Z_VP=Z_VP, score=score, div=div,
                     beta_m=beta_m))

F1 = [r for r in rows if 0.0 < r["Z_PI"] < 150.0]
F2 = [r for r in F1 if 0.45 <= r["score"] <= 0.55]
F3 = sorted(F2, key=lambda r: r["div"])
t_diag = time.time() - t_diag

print("=" * 98)
print(f"{'i':>3} | {'Re(n_eff)':>10} | {'Z0_IBC':>12} | {'Z0_volt_IBC':>12} | "
      f"{'Mode_Score':>10} | {'divergence':>10} | {'verdict':<9}")
print("-" * 98)
for r in sorted(rows, key=lambda q: -q["neff"].real):
    verdict = ("SELECTED" if F3 and r is F3[0] else
               "pass F2" if r in F2 else "cut F2" if r in F1 else "cut F1")
    fmt = (lambda v: f"{v:12.4f}" if abs(v) < 1e5 else f"{v:12.3e}")
    print(f"{r['index']:3d} | {r['neff'].real:10.5f} | {fmt(r['Z_PI'])} | {fmt(r['Z_VP'])} | "
          f"{r['score']:10.5f} | {r['div']*100:9.3f}% | {verdict:<9}")
print("=" * 98)
print(f"filter 1  0 < Z0_IBC < 150 ohm        : {len(rows):3d} -> {len(F1):3d}")
print(f"filter 2  0.45 <= Mode_Score <= 0.55  : {len(F1):3d} -> {len(F2):3d}")
print(f"filter 3  min |Z0_IBC - Z0_voltage|   : {len(F2):3d} -> {min(len(F2),1):3d}    ({t_diag:.1f} s)")
if not F3:
    raise RuntimeError("no quasi-TEM CPW mode survived; raise NUM_MODES or check the shift")
fundamental = F3[0]

# ---- mode gallery ----------------------------------------------------------
_show = [r for r in sorted(rows, key=lambda q: -q["neff"].real)][:8]
if _show:
    _cols = 4
    _rws = int(np.ceil(len(_show) / _cols))
    fig, axes = plt.subplots(_rws, _cols, figsize=(16, 3.2 * _rws))
    axes = np.atleast_1d(axes).flatten()
    for ax, r in zip(axes, _show):
        fl = basis_rf.interpolate(xs[:, r["index"]])
        Ex = np.mean(np.abs(np.asarray(fl[0])[0]) ** 2, axis=-1)
        Ex = Ex / (Ex.max() if Ex.max() > 0 else 1.0)
        mesh.plot(np.sqrt(Ex), ax=ax, shading="flat", cmap="inferno")
        tag = ("SELECTED" if F3 and r is F3[0] else
               "pass F2" if r in F2 else "cut F2" if r in F1 else "cut F1")
        ax.set_title(f"mode {r['index']}: n = {r['neff'].real:.4f}\n"
                     f"Z0 = {r['Z_PI']:.3g} | score = {r['score']:.3f}\n{tag}", fontsize=8)
        ax.set_xlim([-x_go - 10, x_go + 10]); ax.set_ylim([-8, au_h + 10])
        ax.set_aspect("equal")
    for ax in axes[len(_show):]:
        fig.delaxes(ax)
    plt.suptitle(r"$|E_x|$ of every candidate mode (each panel normalised)", y=1.01)
    plt.tight_layout(); plt.show()

In [ ]:
# %% [4] Figures of Merit: n_m, Z0, RF Attenuation
i_sel  = fundamental["index"]
x_sel  = xs[:, i_sel]
beta_m = fundamental["beta_m"]
P_sel  = fundamental["P"]

fields = basis_rf.interpolate(x_sel)
J2     = float(np.real(contour_mode(f_Js2, "all", x_sel, beta_m)))     # contour |Js|^2 dl
P_diel = float(np.real(f_dielectric.assemble(basis_rf, et=fields[0], ez=fields[1],
                                             sig=basis_eps.interpolate(sigma))))

# NOTE: the IBC R' and L_int below are kept for comparison. On a volume mesh the
# corner contribution converges only as h^(1/3) (R' ~10-14 % low at CORNER_RES =
# 0.05 um), and the converged IBC limit itself over-counts the corner current.
# Cell [6] computes R' and L_int from the current inside the gold instead.
# ---------------------------------------------------------------------------
# First-order IBC correction.
#
# The eigensolve above uses PEC electrodes, so it returns the EXTERNAL
# inductance only.  A real conductor presents Zs = (1 + j) * Rs, and the very
# same contour integral that produces the loss resistance also produces an
# internal (surface) inductance:
#
#     R' + j*w*L_int = Zs * contour(|Js/I|^2 dl)   ->   L_int = R'/w
#
# n_m and Z0 both scale with sqrt(L_ext + L_int), so for thick electrodes over
# a narrow gap this is a several-percent shift, not a rounding error.
# ---------------------------------------------------------------------------
G_contour = J2 / fundamental["I_sig"] ** 2          # 1/m
R_prime   = Rs_au * G_contour                       # ohm/m
n_m_pec   = fundamental["neff"].real
Z0_pec    = fundamental["Z_PI"]
L_ext     = Z0_pec * n_m_pec / C0                   # H/m
L_int     = R_prime / omega                         # H/m
kappa     = np.sqrt(1.0 + L_int / L_ext)

n_m  = n_m_pec * kappa
Z0   = Z0_pec * kappa
Z_VP = fundamental["Z_VP"] * kappa
Z_VI = fundamental["V_gap"] / fundamental["I_sig"] * kappa

alpha_c  = R_prime / (2.0 * Z0)                     # Np/m, conductor (IBC) loss
alpha_d  = P_diel / (2.0 * P_sel)                   # Np/m, dielectric loss
alpha_np = alpha_c + alpha_d

alpha_c_db  = DB_PER_NEPER * alpha_c / 100.0
alpha_d_db  = DB_PER_NEPER * alpha_d / 100.0
alpha_db_cm = DB_PER_NEPER * alpha_np / 100.0
neff_im     = -alpha_np / (2 * np.pi * f0 / C0)     # COMSOL emw.neff imaginary part
S21_db      = -DB_PER_NEPER * alpha_np * device_l

# mesh-quality probe: the direct contour integral of Jsz has to reproduce the
# charge-conservation current.  A ratio away from 1 means the conductor contour
# (in particular its re-entrant corners) is still under-resolved.
I_direct = abs(contour_mode(f_Jsz, "sig", x_sel, beta_m))

print("=" * 66)
print("           2D RF BASELINE  --  unetched CPW at 60 GHz")
print("=" * 66)
print(f"  selected mode                      {i_sel:>14d}")
print(f"  emw.neff                    {n_m:>12.5f} {neff_im:+.6f}i")
print(f"  microwave index   n_m              {n_m:>14.5f}")
print(f"  Z0_IBC          (power-current)    {Z0:>14.4f} ohm")
print(f"  Z0_voltage_IBC  (voltage-power)    {Z_VP:>14.4f} ohm")
print(f"  Z0              (voltage-current)  {Z_VI:>14.4f} ohm")
print(f"  Mode_Score                         {fundamental['score']:>14.5f}")
print(f"  quasi-TEM divergence               {fundamental['div']*100:>13.3f} %")
print("-" * 66)
print(f"  conductor loss (IBC)               {alpha_c_db:>14.4f} dB/cm")
print(f"  dielectric loss                    {alpha_d_db:>14.4f} dB/cm")
print(f"  Attenuation                        {alpha_db_cm:>14.4f} dB/cm")
print(f"  S21 over {device_l*1e3:>4.1f} mm                    {S21_db:>14.4f} dB")
print("-" * 66)
print(f"  R'                                 {R_prime:>14.2f} ohm/m")
print(f"  L_ext                              {L_ext*1e9:>14.3f} nH/m")
print(f"  L_int (IBC surface inductance)     {L_int*1e9:>14.3f} nH/m")
print(f"  kappa = sqrt(1 + L_int/L_ext)      {kappa:>14.4f}")
print("-" * 66)
print(f"{'':<28}{'PEC solve':>12}{'quasi-static':>14}{'IBC corr.':>12}")
print(f"{'  n_m':<28}{n_m_pec:>12.5f}{n_m_qs:>14.5f}{n_m:>12.5f}")
print(f"{'  Z0 [ohm]':<28}{Z0_pec:>12.4f}{Z0_qs:>14.4f}{Z0:>12.4f}")
print(f"{'  alpha_c [dB/cm]':<28}{DB_PER_NEPER*R_prime/(2*Z0_pec)/100:>12.4f}"
      f"{alpha_qs_db:>14.4f}{alpha_c_db:>12.4f}")
print(f"{'  |Isig| contour / exact':<28}{I_direct/fundamental['I_sig']:>12.4f}")
print("=" * 66)

# ---- selected mode and its surface current ---------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.4))

_fl = basis_rf.interpolate(x_sel)
_Ex = np.mean(np.abs(np.asarray(_fl[0])[0]) ** 2, axis=-1)
_Ex = np.sqrt(_Ex / (_Ex.max() if _Ex.max() > 0 else 1.0))
mesh.plot(_Ex, ax=axes[0], shading="flat", cmap="inferno", colorbar=True)
axes[0].set_xlim([-x_gi - 10, x_gi + 10]); axes[0].set_ylim([-3, au_h + 4])
axes[0].set_aspect("equal")
axes[0].set_title(rf"selected mode $|E_x|$, $n_m$ = {n_m:.4f}")

_px, _py, _pj = [], [], []
for _fac, _side, _nrm in CONTOUR["all"]:
    _fb = InteriorFacetBasis(mesh, element_rf, facets=_fac, side=_side)
    _f = _fb.interpolate(x_sel)
    _dn = (np.asarray(_f[1].grad[0]) * _nrm[0][:, None]
           + np.asarray(_f[1].grad[1]) * _nrm[1][:, None]) / UM
    _en = (np.asarray(_f[0])[0] * _nrm[0][:, None]
           + np.asarray(_f[0])[1] * _nrm[1][:, None])
    _jz = (-_dn + 1j * beta_m * _en) / (-1j * omega * MU0)
    _mid = mesh.p[:, mesh.facets[:, _fac]].mean(axis=1)
    _px.append(_mid[0]); _py.append(_mid[1])
    _pj.append(np.abs(_jz).mean(axis=1) / fundamental["I_sig"])
_px, _py, _pj = np.concatenate(_px), np.concatenate(_py), np.concatenate(_pj)
_sc = axes[1].scatter(_px, _py, c=np.log10(np.maximum(_pj, 1e-12)), s=6, cmap="viridis")
plt.colorbar(_sc, ax=axes[1], label=r"$\log_{10}|J_{sz}/I|$  (1/m)")
axes[1].set_xlim([-x_gi - 10, x_gi + 10]); axes[1].set_ylim([-3, au_h + 4])
axes[1].set_aspect("equal")
axes[1].set_title("surface current on the IBC contour\n(the corners drive $R'$)")
for ax in axes:
    ax.set_xlabel(r"$x$ ($\mu$m)"); ax.set_ylabel(r"$y$ ($\mu$m)")
plt.tight_layout(); plt.show()

In [ ]:
# %% [5] Cross-check against the recorded COMSOL sweep
# Rows copied verbatim from the corrected 500-LHS COMSOL RF output
# (FINAL_DATASET_nmZ0_alpha_new.xlsx).  An earlier export of the same sweep
# carried an inflated attenuation column and is superseded by this one; if a
# reference row here disagrees with an older table, this one is the right one.
# The first five rows are the original anchors; the rest bracket the design
# space (auc_w 8-80, gap 4.1-20, au_h 1-15, cap_w 1.4-15, wg_h 0.10-0.35).
#   columns: auc_w  gap  au_h  cap_w  wg_h | Re(neff)  Z0_IBC  Z0_voltage_IBC  Mode_Score  Attenuation
COMSOL_ROWS = [
    (60.952,  4.448, 13.336,  3.340, 0.267, 1.65277, 21.5855, 21.4999, 0.49823, 4.39295),
    (78.739,  5.662, 13.680,  3.145, 0.319, 1.68009, 23.5443, 23.3862, 0.49775, 3.51817),
    (63.592,  5.053, 10.779,  3.152, 0.283, 1.71652, 24.7213, 24.5936, 0.49802, 3.88527),
    (75.167,  6.486, 12.843,  2.751, 0.179, 1.74925, 25.2278, 25.0621, 0.49780, 3.21756),
    (78.024,  7.285, 14.935,  5.671, 0.280, 1.72163, 25.4456, 25.2575, 0.49759, 2.92163),
    ( 8.011,  6.820, 14.028,  3.798, 0.220, 1.50951, 37.7845, 37.7902, 0.49920, 3.44720),
    (79.683, 13.729,  3.198,  6.626, 0.253, 2.06870, 40.9653, 40.5784, 0.49771, 1.90493),
    (49.485,  4.098,  7.693,  2.525, 0.329, 1.72715, 26.2526, 26.1627, 0.49826, 4.54752),
    (50.995, 19.998,  7.615,  4.543, 0.124, 2.00253, 46.9215, 46.5939, 0.49768, 1.56550),
    (56.365, 13.964,  1.006,  9.639, 0.314, 2.09527, 47.5107, 47.1921, 0.49807, 1.97977),
    (16.961, 14.051, 14.994,  2.996, 0.350, 1.63038, 48.0871, 48.0078, 0.49843, 1.90944),
    (10.625, 17.794,  4.964, 14.997, 0.238, 1.88020, 71.5774, 71.4678, 0.49877, 2.15844),
    (78.509,  9.642, 12.150,  6.779, 0.100, 1.85910, 29.7362, 29.4948, 0.49754, 2.41693),
]

here = np.array([auc_w, gap, au_h, cap_w, wg_h])
match = None
for row in COMSOL_ROWS:
    if np.allclose(here, np.array(row[:5]), rtol=2e-3, atol=2e-3):
        match = row
        break

print("=" * 74)
print("  COMSOL CROSS-CHECK")
print("=" * 74)
if match is None:
    print("  the geometry in cell [0] is not one of the recorded rows; set it to")
    print("  one of these (auc_w, gap, au_h, cap_w, wg_h) and re-run cells [0]-[4]:")
    for row in COMSOL_ROWS:
        print(f"      {row[0]:7.3f} {row[1]:6.3f} {row[2]:7.3f} {row[3]:6.3f} {row[4]:6.3f}"
              f"   ->  n_m {row[5]:.4f}, Z0 {row[6]:.3f} ohm, alpha {row[9]:.4f} dB/cm")
else:
    _, _, _, _, _, n_ref, z_ref, zv_ref, sc_ref, a_ref = match
    print(f"{'quantity':<28}{'this code':>12}{'COMSOL':>12}{'delta':>12}")
    print("-" * 74)
    for label, mine, ref in [
            ("n_m",                     n_m,   n_ref),
            ("Z0_IBC [ohm]",            Z0,    z_ref),
            ("Z0_voltage_IBC [ohm]",    Z_VP,  zv_ref),
            ("Mode_Score",              fundamental["score"], sc_ref),
            ("Attenuation [dB/cm]",     alpha_db_cm, a_ref)]:
        print(f"{'  '+label:<28}{mine:>12.5f}{ref:>12.5f}{(mine/ref-1)*100:>11.2f}%")
    print("-" * 74)
    print(f"{'  L (= Z0*n_m/c) [nH/m]':<28}{Z0*n_m/C0*1e9:>12.4f}"
          f"{z_ref*n_ref/C0*1e9:>12.4f}{(Z0*n_m)/(z_ref*n_ref)*100-100:>11.2f}%")
    print(f"{'  C (= n_m/(Z0*c)) [pF/m]':<28}{n_m/(Z0*C0)*1e12:>12.4f}"
          f"{n_ref/(z_ref*C0)*1e12:>12.4f}{(n_m*z_ref)/(Z0*n_ref)*100-100:>11.2f}%")
print("=" * 74)

In [ ]:
# %% [6] Conductor loss from the current INSIDE the gold (replaces the IBC R' and L_int)
#
# Why: the IBC of cell [4] needs the PEC surface current at the electrode
# corners, where it is singular (J ~ r^-1/3). The contour integral is finite,
# but on a volume mesh it converges only as h^(1/3): every halving of the corner
# element adds ~2-3 % to R', and at CORNER_RES = 0.05 um R' is ~10-14 % low.
# Even fully converged (1e-7 um corner panels, boundary-element check), the IBC
# limit is not the true value either: it treats every point of the corner as a
# flat half-space and over-counts the corner current by a few percent.
#
# This cell removes the surface-impedance approximation instead of refining it:
# it solves for the longitudinal current density J_z(x, y) inside the electrodes
# (skin AND proximity effect, valid from DC to the strong skin effect), on a
# tensor mesh graded to MQS_HFINE at every metal surface:
#
#     -div( grad A_z / mu0 ) + j w sigma A_z + sigma V'_k = 0     in electrode k
#     -div( grad A_z / mu0 )                       = 0     outside
#     int_k sigma (-j w A_z - V'_k) dS = I_k     (signal +1, each ground -1/2)
#
#     R' + j w L' = V'_gnd - V'_sig
#
# Displacement current inside the gold is 1e-7 of the conduction current, and the
# electrodes are 1e-3 wavelengths wide, so this is Maxwell's equations without
# further approximation. Dielectrics do not enter (they do not change B).
# L_int is the difference between L' and the external (PEC) inductance of the
# same mesh and box, so the box size cancels out. Half domain x >= 0: A_z is
# even in x, x = 0 is a natural boundary, A_z = 0 on the far box.
import scipy.sparse as _sp
import scipy.sparse.linalg as _spla
from skfem import ElementTriP2, LinearForm

MQS_HFINE = 0.03       # um, element size at every metal surface (skin depth / 10 at 60 GHz)
MQS_BOX   = 600.0      # um, half size of the A = 0 box


def _graded(a, b, ha, hb, hmax, r=1.15):
    L = b - a
    left, h = [0.0], ha
    while left[-1] < L / 2:
        left.append(left[-1] + h); h = min(h * r, hmax)
    right, h = [0.0], hb
    while right[-1] < L / 2:
        right.append(right[-1] + h); h = min(h * r, hmax)
    left, right = np.array(left), L - np.array(right)
    pts = np.unique(np.concatenate([[0.0, L], left[left < L / 2], right[right >= L / 2]]))
    keep = [pts[0]]
    for q in pts[1:]:
        if q - keep[-1] > 0.3 * min(ha, hb):
            keep.append(q)
    keep[-1] = L
    return a + np.array(keep)


def _axis(breaks, hfine, hmax_metal, hmax_air, metal_spans):
    out = []
    for a, b in zip(breaks[:-1], breaks[1:]):
        inside = any(lo <= a and b <= hi for lo, hi in metal_spans)
        hmax = hmax_metal if inside else hmax_air
        ha = hfine if a != breaks[0] else hmax_air
        hb = hfine if b != breaks[-1] else hmax_air
        if a == 0.0 and breaks[0] == 0.0:       # x = 0 symmetry plane: no surface there
            ha = min(hmax, 0.5)
        out.append(_graded(a, b, ha, hb, hmax))
    return np.unique(np.concatenate(out))


def conductor_impedance(f, hfine=MQS_HFINE, box=MQS_BOX, sigma_au=SIGMA_AU):
    """R' (ohm/m), L' (H/m) and the PEC external L_ext of the same box, for the
    electrodes of cell [0] (their y offset does not matter)."""
    xa = _axis([0.0, x_sig, x_gi, x_go, box], hfine, 0.5, 40.0, [(0.0, x_sig), (x_gi, x_go)])
    ya = _axis([-box, 0.0, au_h, box], hfine, 0.5, 40.0, [(0.0, au_h)])
    m = MeshTri.init_tensor(xa * UM, ya * UM)
    c = m.p[:, m.t].mean(axis=1) / UM
    inside = (c[1] > 0) & (c[1] < au_h)
    cond = np.where(inside & (c[0] < x_sig), 1, np.where(inside & (c[0] > x_gi) & (c[0] < x_go), 2, 0))
    b2 = Basis(m, ElementTriP2())
    b0 = b2.with_element(ElementTriP0())

    @BilinearForm
    def _stiff(u, v, w):
        return (grad(u)[0] * grad(v)[0] + grad(u)[1] * grad(v)[1]) / MU0

    @BilinearForm
    def _mass(u, v, w):
        return w.s * u * v

    @LinearForm
    def _lf(v, w):
        return w.s * v

    K = asm(_stiff, b2)
    Ms = asm(_mass, b2, s=b0.interpolate(np.where(cond > 0, sigma_au, 0.0)))
    pp = m.p[:, m.t]
    area = 0.5 * np.abs((pp[0, 1] - pp[0, 0]) * (pp[1, 2] - pp[1, 0])
                        - (pp[0, 2] - pp[0, 0]) * (pp[1, 1] - pp[1, 0]))
    Bm = np.column_stack([asm(_lf, b2, s=b0.interpolate((cond == k) * sigma_au)) for k in (1, 2)])
    Gk = np.array([sigma_au * area[cond == k].sum() for k in (1, 2)])
    xm, ymn, ymx = m.p[0].max(), m.p[1].min(), m.p[1].max()
    D = b2.get_dofs(lambda X: (np.abs(X[0] - xm) < 1e-12) | (np.abs(X[1] - ymn) < 1e-12)
                    | (np.abs(X[1] - ymx) < 1e-12)).flatten()
    keep = np.setdiff1d(np.arange(b2.N), D)
    Kk, Mk = K.tocsr()[keep][:, keep], Ms.tocsr()[keep][:, keep]
    Bd = Bm[keep]
    w = 2 * np.pi * f
    # [K + jwM   B] [A ]   [ 0]       solved through its Schur complement on V'
    # [jw B^T    G] [V'] = [-I]       (one sparse LU, then a 2 x 2 system)
    lu = _spla.splu((Kk + 1j * w * Mk).tocsc(), permc_spec="MMD_AT_PLUS_A")
    X = lu.solve(Bd.astype(complex))
    V = np.linalg.solve(np.diag(Gk) - 1j * w * (Bd.T @ X),
                        -np.array([0.5, -0.5]))               # half domain: +1/2 and -1/2
    Z = -(V[0] - V[1])                                          # E = -j w A - V'
    # external inductance of the same box: PEC electrodes (A pinned to 1 and 0
    # on the gold), current from the residual over the pinned DOFs, L = 1 / I
    d_sig = np.unique(b2.element_dofs[:, cond == 1])
    d_gnd = np.unique(b2.element_dofs[:, cond == 2])
    DD = np.unique(np.concatenate([D, d_sig, d_gnd]))
    a_D = np.zeros(b2.N); a_D[d_sig] = 1.0
    Kc, fc, ac, II = condense(K, np.zeros(b2.N), x=a_D, D=DD)
    a = ac.copy(); a[II] = solve(Kc, fc)
    I_half = float((K @ a)[d_sig].sum())                        # current of the half signal
    L_ext_box = 1.0 / (2.0 * I_half)
    return Z.real, Z.imag / w, L_ext_box, dict(n_el=m.nelements, ndof=b2.N)


t_mqs = time.time()
R_mqs, L_mqs, L_ext_box, mqs_info = conductor_impedance(f0)
L_int_mqs = L_mqs - L_ext_box
t_mqs = time.time() - t_mqs

# same post-processing as cell [4], with the conductor-interior R' and L_int
kappa_m   = np.sqrt(1.0 + L_int_mqs / L_ext)
n_m_c     = n_m_pec * kappa_m
Z0_c      = Z0_pec * kappa_m
Z_VP_c    = fundamental["Z_VP"] * kappa_m
alpha_c_c = R_mqs / (2.0 * Z0_c)
alpha_db_cm_c = DB_PER_NEPER * (alpha_c_c + alpha_d) / 100.0
R_dc_exact = 1.0 / (SIGMA_AU * auc_w * au_h * UM ** 2) + 0.5 / (SIGMA_AU * aul_w * au_h * UM ** 2)

print("=" * 74)
print("  CONDUCTOR-INTERIOR SOLVE  (skin + proximity effect, no IBC)")
print("=" * 74)
print(f"  mesh {mqs_info['n_el']} triangles, {mqs_info['ndof']} P2 DOF, h = {MQS_HFINE} um at the metal, "
      f"{t_mqs:.1f} s")
print(f"  R_dc (exact, for reference)        {R_dc_exact:14.2f} ohm/m")
print(f"{'':<30}{'IBC [4]':>12}{'interior':>12}{'delta':>10}")
print("-" * 74)
for label, a_, b_ in [("R' [ohm/m]", R_prime, R_mqs),
                      ("L_int [nH/m]", L_int * 1e9, L_int_mqs * 1e9),
                      ("n_m", n_m, n_m_c),
                      ("Z0_IBC [ohm]", Z0, Z0_c),
                      ("Z0_voltage [ohm]", Z_VP, Z_VP_c),
                      ("conductor loss [dB/cm]", alpha_c_db, DB_PER_NEPER * alpha_c_c / 100),
                      ("Attenuation [dB/cm]", alpha_db_cm, alpha_db_cm_c)]:
    print(f"  {label:<28}{a_:>12.4f}{b_:>12.4f}{(b_ / a_ - 1) * 100:>9.2f}%")
print("=" * 74)
if match is not None:
    print(f"  COMSOL row (IBC): attenuation {match[9]:.4f} dB/cm  ->  conductor-interior "
          f"{alpha_db_cm_c:.4f} dB/cm ({(alpha_db_cm_c / match[9] - 1) * 100:+.1f} %)")
